In [1]:
import pandas as pd

crime = pd.read_csv("data/cleaned_crime_long.csv", parse_dates=["Date"])

borough_monthly = (
    crime.groupby(["Borough", "Date"])["CrimeCount"]
    .sum()
    .reset_index()
    .rename(columns={"CrimeCount": "TotalCrimes"})
    .sort_values(["Borough", "Date"])
)

print(borough_monthly.shape)
print(borough_monthly.head())

(6112, 3)
                Borough       Date  TotalCrimes
0  Barking and Dagenham 2010-04-01         1628
1  Barking and Dagenham 2010-05-01         1612
2  Barking and Dagenham 2010-06-01         1713
3  Barking and Dagenham 2010-07-01         1739
4  Barking and Dagenham 2010-08-01         1636


In [2]:
borough_avg = borough_monthly.groupby("Borough")["TotalCrimes"].transform("mean")
borough_monthly["CrimeDensity"] = borough_monthly["TotalCrimes"] / borough_avg


borough_monthly["Crimes_Lag1"] = borough_monthly.groupby("Borough")["TotalCrimes"].shift(1)
borough_monthly["Crimes_Lag12"] = borough_monthly.groupby("Borough")["TotalCrimes"].shift(12)


borough_monthly["Crimes_RollingMean3"] = (
    borough_monthly.groupby("Borough")["TotalCrimes"]
    .shift(1)
    .rolling(window=3)
    .mean()
    .reset_index(level=0, drop=True)
)

borough_monthly = borough_monthly.dropna(subset=["Crimes_Lag1", "Crimes_Lag12", "Crimes_RollingMean3"])

print(borough_monthly.shape)
print(borough_monthly.head())

(5728, 7)
                 Borough       Date  TotalCrimes  CrimeDensity  Crimes_Lag1  \
12  Barking and Dagenham 2011-04-01         1585      1.036668       1581.0   
13  Barking and Dagenham 2011-05-01         1702      1.113192       1585.0   
14  Barking and Dagenham 2011-06-01         1593      1.041900       1702.0   
15  Barking and Dagenham 2011-07-01         1591      1.040592       1593.0   
16  Barking and Dagenham 2011-08-01         1650      1.079181       1591.0   

    Crimes_Lag12  Crimes_RollingMean3  
12        1628.0          1475.333333  
13        1612.0          1517.333333  
14        1713.0          1622.666667  
15        1739.0          1626.666667  
16        1636.0          1628.666667  


In [3]:
borough_monthly["RiskLevel"] = pd.qcut(
    borough_monthly["CrimeDensity"], q=3, labels=["Low", "Medium", "High"]
)

print(borough_monthly["RiskLevel"].value_counts())
print(borough_monthly[["Borough", "Date", "CrimeDensity", "RiskLevel"]].head())

RiskLevel
Low       1910
Medium    1909
High      1909
Name: count, dtype: int64
                 Borough       Date  CrimeDensity RiskLevel
12  Barking and Dagenham 2011-04-01      1.036668    Medium
13  Barking and Dagenham 2011-05-01      1.113192      High
14  Barking and Dagenham 2011-06-01      1.041900    Medium
15  Barking and Dagenham 2011-07-01      1.040592    Medium
16  Barking and Dagenham 2011-08-01      1.079181      High


In [4]:
cutoff_date = borough_monthly["Date"].max() - pd.DateOffset(months=12)

train = borough_monthly[borough_monthly["Date"] < cutoff_date]
test = borough_monthly[borough_monthly["Date"] >= cutoff_date]

feature_cols = ["Crimes_Lag1", "Crimes_Lag12", "Crimes_RollingMean3"]

X_train, y_train = train[feature_cols], train["RiskLevel"]
X_test, y_test = test[feature_cols], test["RiskLevel"]

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)
print("Cutoff date:", cutoff_date.date())

Train size: (5312, 3)
Test size: (416, 3)
Cutoff date: 2025-02-01


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

pred = log_reg.predict(X_test_scaled)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

        High       0.81      0.41      0.55       280
         Low       0.08      0.90      0.15        20
      Medium       0.31      0.15      0.20       116

    accuracy                           0.36       416
   macro avg       0.40      0.49      0.30       416
weighted avg       0.64      0.36      0.43       416



In [6]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=5, random_state=42)
tree.fit(X_train, y_train)

pred = tree.predict(X_test)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

        High       0.73      0.65      0.68       280
         Low       0.08      0.65      0.15        20
      Medium       0.42      0.04      0.08       116

    accuracy                           0.48       416
   macro avg       0.41      0.45      0.30       416
weighted avg       0.61      0.48      0.49       416



In [7]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
forest.fit(X_train, y_train)

pred = forest.predict(X_test)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

        High       0.80      0.61      0.69       280
         Low       0.09      0.60      0.16        20
      Medium       0.43      0.25      0.32       116

    accuracy                           0.51       416
   macro avg       0.44      0.49      0.39       416
weighted avg       0.66      0.51      0.56       416



In [9]:
pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 108.3 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.4/303.4 MB 107.8 MB/s  0:00:020:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]m1/2 [xgboost]
Note: you may need to restart the kernel to use updated packages.


In [10]:
from xgboost import XGBClassifier

label_map = {"Low": 0, "Medium": 1, "High": 2}
y_train_xgb = y_train.map(label_map)
y_test_xgb = y_test.map(label_map)

xgb = XGBClassifier(n_estimators=200, max_depth=5, random_state=42, eval_metric="mlogloss")
xgb.fit(X_train, y_train_xgb)

pred = xgb.predict(X_test)

print(classification_report(y_test_xgb, pred, target_names=["Low", "Medium", "High"]))

              precision    recall  f1-score   support

         Low       0.07      0.40      0.12        20
      Medium       0.37      0.41      0.39       116
        High       0.80      0.48      0.60       280

    accuracy                           0.46       416
   macro avg       0.41      0.43      0.37       416
weighted avg       0.65      0.46      0.52       416



In [11]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree", "Random Forest", "XGBoost"],
    "Accuracy": [0.36, 0.48, 0.51, 0.46],
    "Weighted F1": [0.43, 0.49, 0.56, 0.52]
})

print(comparison.sort_values("Weighted F1", ascending=False))

                 Model  Accuracy  Weighted F1
2        Random Forest      0.51         0.56
3              XGBoost      0.46         0.52
1        Decision Tree      0.48         0.49
0  Logistic Regression      0.36         0.43
